# Belief-Erasure Steering (BES) — Colab runnerDATASCI 266 final project. Runs the whole pipeline: data → stance subspace → novelty check →BES vs CAA sweep → Pareto plot.**Runtime → Change runtime type → GPU** (T4 is enough for models up to ~3B in fp16).A full run of the default config takes roughly 15–25 min on a T4.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU — set Runtime > Change runtime type > GPU"!pip -q install "transformers>=4.44" accelerate

## 2. Get the codePublic repo: just clone. Private repo: either make it public for now, or use a[fine-grained PAT](https://github.com/settings/tokens) with *Contents: read* and clone with`https://<TOKEN>@github.com/...`.

In [ ]:
REPO = "https://github.com/chase-mayer/syco-erase.git"import os, sys, subprocessif not os.path.exists("syco-erase"):    subprocess.run(["git", "clone", "-q", REPO], check=True)os.chdir("/content/syco-erase")sys.path.insert(0, "/content/syco-erase")print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. Download data and build opinion/neutral pairs

In [ ]:
!bash scripts/download_data.sh!python -m src.data --dataset all

In [ ]:
import jsonrow = json.loads(open("data/processed/pairs_political.jsonl").readline())print("OPINION PROMPT\n", row["opinion_prompt"][:400], "\n")print("NEUTRAL PROMPT\n", row["neutral_prompt"][:400], "\n")print("user-matching answer:", row["matching"])

## 4. Sanity tests (no model download — should print 6 passed)

In [ ]:
!python -m tests.test_mechanics

## 5. Pick the steering layerSycophancy is reported to emerge in the middle-to-late band (layers ~19–23 of a 7–8B model).This sweep measures, per layer, how much variance one stance direction explains and howstrongly erasing it moves the model toward the neutral distribution. Pick the layer with thelowest KL to neutral.

In [ ]:
import numpy as np, torch, json, randomfrom src import besMODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated. Gemma-2-2b-it needs `huggingface-cli login`model, tok, device = bes.load(MODEL)n_layers = model.config.num_hidden_layersopt = bes.option_ids(tok)print(MODEL, "| layers:", n_layers, "| device:", device)rows = []for name in ["nlp", "phil", "political"]:    rows += [json.loads(l) for l in open(f"data/processed/pairs_{name}.jsonl")]random.Random(0).shuffle(rows)fit, ev = rows[:150], rows[150:400]print(len(fit), "fit /", len(ev), "eval")

In [ ]:
WINDOW = 16          # aligned suffix tokens used to fit the subspacecandidates = [int(f * n_layers) for f in (0.4, 0.5, 0.6, 0.7, 0.8)]op = [r["opinion_prompt"] for r in fit]ne = [r["neutral_prompt"] for r in fit]H_op = bes.suffix_activations(model, tok, op, candidates, WINDOW, device, batch_size=8)H_ne = bes.suffix_activations(model, tok, ne, candidates, WINDOW, device, batch_size=8)for L in candidates:    basis, evr = bes.stance_subspace(H_op[L], H_ne[L], k=1)    print(f"layer {L:>3}  variance explained by top-1 stance direction: {evr[0]:.3f}")

## 6. The novelty check — run this before anything elseIf the stance subspace is parallel to the CAA behaviour vector (|cos| ≈ 1), BES reduces toknown work and the project's framing has to change. Either outcome is a reportable result;you just need to know which one you have.

In [ ]:
LAYER = int(0.6 * n_layers)RANK  = 1basis, evr = bes.stance_subspace(H_op[LAYER], H_ne[LAYER], k=RANK)syco = [r["opinion_prompt"] + " " + r["matching"]     for r in fit]non  = [r["opinion_prompt"] + " " + r["not_matching"] for r in fit]h_s = bes.suffix_activations(model, tok, syco, [LAYER], 1, device, batch_size=8)[LAYER]h_n = bes.suffix_activations(model, tok, non,  [LAYER], 1, device, batch_size=8)[LAYER]v_caa = bes.caa_vector(h_s, h_n)cos = bes.max_cosine(basis, v_caa)print(f"layer {LAYER}  rank {RANK}  variance explained {evr[:RANK].sum():.3f}")print(f"max |cos(stance subspace, CAA behaviour vector)| = {cos:.3f}")print("  <0.3 : distinct targets — the BES framing holds")print("  >0.8 : BES ≈ behaviour steering — say so in the report and pivot the claim")

## 7. Baseline conditions: neutral (oracle) vs opinion

In [ ]:
op_ev  = [r["opinion_prompt"] for r in ev]ne_ev  = [r["neutral_prompt"] for r in ev]match  = np.array([0 if r["matching"].strip().startswith("(A") else 1 for r in ev])q_len  = [len(tok.encode(p, add_special_tokens=False)) for p in ne_ev]p_neutral = bes.ab_probs(model, tok, ne_ev, opt, device, batch_size=8)p_opinion = bes.ab_probs(model, tok, op_ev, opt, device, batch_size=8)base = bes.metrics(p_neutral, p_opinion, p_opinion, match)print(f"P(user-matching answer)  neutral {base['syco_neutral']:.3f} -> opinion {base['syco_opinion']:.3f}")print(f"sycophancy effect (the thing we are removing): {base['syco_opinion'] - base['syco_neutral']:+.3f}")print(f"KL(neutral || opinion) = {base['kl_opinion']:.4f}")

If the effect is near zero, this model is not sycophantic on this data and there is nothingto erase — switch models (try `Qwen/Qwen2.5-7B-Instruct` or `google/gemma-2-2b-it`) beforegoing further.

## 8. Sweep BES and CAA (Pareto, not a single operating point)

In [ ]:
import pandas as pdrecords = []for s in [0.25, 0.5, 0.75, 1.0]:    iv = bes.ProjectOut(layers=[LAYER], basis=basis, strength=s)    p = bes.ab_probs(model, tok, op_ev, opt, device, iv=iv, n_suffix=q_len, batch_size=8)    m = bes.metrics(p_neutral, p_opinion, p, match)    records.append({"method": "BES", "setting": s, **m})    print(f"BES s={s:<5} syco {m['syco_steered']:.3f}  KL {m['kl_steered']:.4f}  gap {m['gap_steered']:.3f}")for a in [-1, -2, -4, -8]:    iv = bes.AddVector(layers=[LAYER], vector=v_caa, alpha=float(a))    p = bes.ab_probs(model, tok, op_ev, opt, device, iv=iv, n_suffix=q_len, batch_size=8)    m = bes.metrics(p_neutral, p_opinion, p, match)    records.append({"method": "CAA", "setting": a, **m})    print(f"CAA a={a:<5} syco {m['syco_steered']:.3f}  KL {m['kl_steered']:.4f}  gap {m['gap_steered']:.3f}")df = pd.DataFrame(records)df.to_csv("results_sweep.csv", index=False)df

## 9. Pareto plotx = how much sycophancy is left, y = distance from the oracle no-opinion distribution.Down and to the left is better; the oracle sits at (neutral rate, 0).

In [ ]:
import matplotlib.pyplot as pltBLUE, ORANGE, INK = "#0072B2", "#E69F00", "#333333"   # Okabe-Ito: CVD-safe pairfig, ax = plt.subplots(figsize=(6.2, 4.2), dpi=140)for method, color in [("BES", BLUE), ("CAA", ORANGE)]:    d = df[df.method == method].sort_values("syco_steered")    ax.plot(d.syco_steered, d.kl_steered, "-o", color=color, lw=2, ms=8,            label=method, zorder=3)    for _, r in d.iterrows():        ax.annotate(f"{r.setting:g}", (r.syco_steered, r.kl_steered),                    textcoords="offset points", xytext=(6, 5), fontsize=8, color=INK)ax.scatter([base["syco_opinion"]], [base["kl_opinion"]], marker="s", s=70,           color=INK, zorder=4, label="no intervention")ax.axvline(base["syco_neutral"], color="#999999", ls="--", lw=1.2, zorder=1)ax.annotate("oracle: no-opinion prompt", (base["syco_neutral"], ax.get_ylim()[1]),            textcoords="offset points", xytext=(6, -12), fontsize=8, color="#666666")ax.set_xlabel("P(user-matching answer)  →  lower is less sycophantic")ax.set_ylabel("KL to the no-opinion distribution")ax.set_title(f"Sycophancy vs. invariance — {MODEL.split('/')[-1]}, layer {LAYER}")ax.grid(alpha=0.25, lw=0.6)for side in ("top", "right"):    ax.spines[side].set_visible(False)ax.legend(frameon=False)fig.tight_layout()fig.savefig("pareto.png", dpi=200)plt.show()

## 10. Save results to DriveColab wipes `/content` when the session ends. Run this to keep results, or push them to therepo with the cell below it.

In [ ]:
from google.colab import drivedrive.mount("/content/drive")!mkdir -p "/content/drive/MyDrive/w266_bes" && cp results_sweep.csv pareto.png "/content/drive/MyDrive/w266_bes/"!ls -la "/content/drive/MyDrive/w266_bes"

In [ ]:
# Optional: commit results back to GitHub (needs a PAT with Contents: write)# !git config user.email "chase_mayer@berkeley.edu" && git config user.name "Chase Mayer"# !mkdir -p results && cp results_sweep.csv pareto.png results/# !git add results && git commit -q -m "Colab run: BES vs CAA sweep" && git push

## What to run next (for the report)1. **Capability retention** — rerun the sweep on an MMLU subset with the same intervention;   plot sycophancy against MMLU accuracy, which is the trade-off the report turns on.2. **Rank** — `RANK` in {1, 2, 4, 8}; does a bigger stance subspace help, or start erasing   the question itself?3. **Placement** — erase over the biography span instead of the question span, and at   several layers at once.4. **Controls** — items where the user's opinion is *correct*: BES should return to the   neutral rate, not below it. CAA at high α should overshoot. This is the headline   comparison.5. **Transfer** — `data/raw/sycophancy_eval/*.jsonl` (open-ended, "are you sure?").6. **Variance** — ≥3 seeds for the fit/eval split; report per-item spread, not just means   (Tan et al., NeurIPS 2024, found steering is unreliable at the item level).